# DataPulse — Day 19: Advanced SQL JOIN Analysis & Data Validation

This notebook builds on the Day 18 `customer_summary` and `product_summary` tables. It answers business questions with joins while checking that the joins preserve the intended transaction-level grain.

## Learning goals

- Compare `INNER JOIN` and `LEFT JOIN` in a practical customer-sales scenario.
- Join each transaction to customer and product summaries without duplicating rows.
- Validate matched and unmatched customer/product records.
- Understand customer category and product performance, including high-value customer preferences.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

connection = sqlite3.connect('../data/datapulse.db')


In [ ]:
# The Day 18 summaries are the reference layers used in this analysis.
for table_name in ['sales', 'customer_summary', 'product_summary']:
    row_count = pd.read_sql_query(f'SELECT COUNT(*) AS row_count FROM {table_name}', connection)
    print(f'{table_name}: {int(row_count.loc[0, "row_count"]):,} rows')


## 1. INNER JOIN vs LEFT JOIN

An `INNER JOIN` keeps only sales with a matching customer summary. A `LEFT JOIN` keeps all sales rows, which makes it the safer choice for a data-quality check: unmatched sales remain visible with null customer fields.

In [ ]:
inner_join_sample = pd.read_sql_query("""
SELECT s.order_id, s.customer_id, s.product, s.category, s.net_sales,
       c.total_orders AS customer_total_orders,
       c.total_revenue AS customer_lifetime_revenue
FROM sales AS s
INNER JOIN customer_summary AS c ON s.customer_id = c.customer_id
LIMIT 10;
""", connection)
inner_join_sample


In [ ]:
left_join_sample = pd.read_sql_query("""
SELECT s.order_id, s.customer_id, s.product, s.category, s.net_sales,
       c.total_orders AS customer_total_orders,
       c.total_revenue AS customer_lifetime_revenue
FROM sales AS s
LEFT JOIN customer_summary AS c ON s.customer_id = c.customer_id
LIMIT 10;
""", connection)
left_join_sample


## 2. Check row counts before and after joins

A customer summary has one row per customer; therefore, joining it to a transaction should not add transaction rows. The product summary is grouped by both `product` and `category`, so both fields are used in that join. This is the key safeguard against an accidental many-to-many join.

In [ ]:
row_counts = pd.read_sql_query("""
SELECT 'sales_before_join' AS check_name, COUNT(*) AS row_count FROM sales
UNION ALL
SELECT 'inner_join_customer_summary', COUNT(*)
FROM sales AS s INNER JOIN customer_summary AS c ON s.customer_id = c.customer_id
UNION ALL
SELECT 'left_join_customer_summary', COUNT(*)
FROM sales AS s LEFT JOIN customer_summary AS c ON s.customer_id = c.customer_id
UNION ALL
SELECT 'three_table_join_rows', COUNT(*)
FROM sales AS s
INNER JOIN customer_summary AS c ON s.customer_id = c.customer_id
INNER JOIN product_summary AS p ON s.product = p.product AND s.category = p.category
UNION ALL
SELECT 'distinct_order_ids_after_join', COUNT(DISTINCT s.order_id)
FROM sales AS s
INNER JOIN customer_summary AS c ON s.customer_id = c.customer_id
INNER JOIN product_summary AS p ON s.product = p.product AND s.category = p.category;
""", connection)
row_counts


In [ ]:
three_table_sample = pd.read_sql_query("""
SELECT s.order_id, s.customer_id, s.product, s.category, s.region, s.net_sales,
       c.total_revenue AS customer_lifetime_revenue,
       p.total_revenue AS product_lifetime_revenue
FROM sales AS s
INNER JOIN customer_summary AS c ON s.customer_id = c.customer_id
INNER JOIN product_summary AS p ON s.product = p.product AND s.category = p.category
LIMIT 10;
""", connection)
three_table_sample


## 3. Customer-level category and product performance

The query below aggregates transaction measures (`quantity` and `net_sales`) after the join. `MAX(c.total_revenue)` is used only to retain the one customer-level value; summing it would repeat a customer's lifetime revenue once per transaction.

In [ ]:
customer_category_performance = pd.read_sql_query("""
SELECT
    s.customer_id,
    s.category,
    COUNT(DISTINCT s.order_id) AS transaction_count,
    SUM(s.quantity) AS units_sold,
    ROUND(SUM(s.net_sales), 2) AS category_revenue,
    ROUND(MAX(c.total_revenue), 2) AS customer_lifetime_revenue
FROM sales AS s
INNER JOIN customer_summary AS c ON s.customer_id = c.customer_id
INNER JOIN product_summary AS p ON s.product = p.product AND s.category = p.category
GROUP BY s.customer_id, s.category
ORDER BY category_revenue DESC;
""", connection)
customer_category_performance.head(10)


In [ ]:
# A compact, ranked table makes the leading combinations easy to compare.
top_customer_categories = customer_category_performance.head(10).copy()
top_customer_categories['customer_category'] = (
    top_customer_categories['customer_id'] + ' — ' + top_customer_categories['category']
)
top_customer_categories[['customer_category', 'transaction_count', 'units_sold', 'category_revenue']]


In [ ]:
customer_product_performance = pd.read_sql_query("""
SELECT
    s.customer_id,
    s.category,
    s.product,
    COUNT(DISTINCT s.order_id) AS transaction_count,
    SUM(s.quantity) AS units_sold,
    ROUND(SUM(s.net_sales), 2) AS product_revenue
FROM sales AS s
INNER JOIN customer_summary AS c ON s.customer_id = c.customer_id
INNER JOIN product_summary AS p ON s.product = p.product AND s.category = p.category
GROUP BY s.customer_id, s.category, s.product
ORDER BY product_revenue DESC;
""", connection)
customer_product_performance.head(10)


## 4. High-value customers and their preferences

Day 18 defined high-value customers as those with more than ₹50,000 in lifetime revenue. We first retrieve their product-level revenue, then use beginner-friendly pandas sorting and `drop_duplicates` to choose each customer's highest-revenue product and category. Ties are resolved alphabetically for reproducibility.

In [ ]:
high_value_product_detail = pd.read_sql_query("""
SELECT
    s.customer_id,
    s.category,
    s.product,
    COUNT(DISTINCT s.order_id) AS transaction_count,
    SUM(s.quantity) AS units_sold,
    ROUND(SUM(s.net_sales), 2) AS product_revenue,
    ROUND(MAX(c.total_revenue), 2) AS customer_lifetime_revenue
FROM sales AS s
INNER JOIN customer_summary AS c ON s.customer_id = c.customer_id
INNER JOIN product_summary AS p ON s.product = p.product AND s.category = p.category
WHERE c.total_revenue > 50000
GROUP BY s.customer_id, s.category, s.product
ORDER BY s.customer_id, product_revenue DESC;
""", connection)

preferred_products = (
    high_value_product_detail
    .sort_values(['customer_id', 'product_revenue', 'product'], ascending=[True, False, True])
    .drop_duplicates('customer_id')
    [['customer_id', 'category', 'product', 'product_revenue', 'customer_lifetime_revenue']]
    .rename(columns={'category': 'preferred_product_category', 'product': 'preferred_product'})
)

category_preferences = (
    high_value_product_detail
    .groupby(['customer_id', 'category'], as_index=False)['product_revenue'].sum()
    .rename(columns={'product_revenue': 'category_revenue'})
    .sort_values(['customer_id', 'category_revenue', 'category'], ascending=[True, False, True])
    .drop_duplicates('customer_id')
    .rename(columns={'category': 'preferred_category'})
)

high_value_customer_preferences = (
    preferred_products
    .merge(category_preferences[['customer_id', 'preferred_category', 'category_revenue']], on='customer_id')
    .sort_values('customer_lifetime_revenue', ascending=False)
)
high_value_customer_preferences.head(10)


## 5. JOIN-based data-quality validation

`LEFT JOIN` plus `IS NULL` identifies sales records missing their customer or product-summary match. The validation query counts both types of mismatch while preserving all sales rows.

In [ ]:
join_data_quality_validation = pd.read_sql_query("""
SELECT
    COUNT(*) AS sales_rows,
    SUM(CASE WHEN c.customer_id IS NULL THEN 1 ELSE 0 END) AS unmatched_customer_rows,
    SUM(CASE WHEN p.product IS NULL THEN 1 ELSE 0 END) AS unmatched_product_rows,
    SUM(CASE WHEN c.customer_id IS NOT NULL AND p.product IS NOT NULL THEN 1 ELSE 0 END) AS fully_matched_rows
FROM sales AS s
LEFT JOIN customer_summary AS c ON s.customer_id = c.customer_id
LEFT JOIN product_summary AS p ON s.product = p.product AND s.category = p.category;
""", connection)
join_data_quality_validation


In [ ]:
unmatched_customers = pd.read_sql_query("""
SELECT s.customer_id, COUNT(*) AS unmatched_transaction_rows, ROUND(SUM(s.net_sales), 2) AS unmatched_revenue
FROM sales AS s
LEFT JOIN customer_summary AS c ON s.customer_id = c.customer_id
WHERE c.customer_id IS NULL
GROUP BY s.customer_id
ORDER BY unmatched_transaction_rows DESC;
""", connection)
unmatched_customers


## Automated Day 19 insights

The next cell derives its statements from the query outputs so the notebook stays accurate when the data changes.

In [ ]:
sales_rows = int(row_counts.loc[row_counts['check_name'] == 'sales_before_join', 'row_count'].iloc[0])
three_table_rows = int(row_counts.loc[row_counts['check_name'] == 'three_table_join_rows', 'row_count'].iloc[0])
distinct_orders = int(row_counts.loc[row_counts['check_name'] == 'distinct_order_ids_after_join', 'row_count'].iloc[0])
quality = join_data_quality_validation.iloc[0]
top_combination = customer_category_performance.iloc[0]
preference_counts = high_value_customer_preferences['preferred_category'].value_counts()
top_preferred_category = preference_counts.index[0]
top_preferred_category_count = int(preference_counts.iloc[0])
top_product_counts = high_value_customer_preferences['preferred_product'].value_counts()
top_preferred_product = top_product_counts.index[0]
top_preferred_product_count = int(top_product_counts.iloc[0])

print('=' * 70)
print('DATAPULSE — DAY 19 SQL JOIN INSIGHTS')
print('=' * 70)
print(f'1. Row preservation: sales has {sales_rows:,} rows; the three-table join also has {three_table_rows:,} rows and {distinct_orders:,} distinct order IDs.')
print(f'2. Data quality: {int(quality.unmatched_customer_rows):,} sales rows lack a customer-summary match and {int(quality.unmatched_product_rows):,} lack a product-summary match.')
print(f'3. Largest customer-category result: {top_combination.customer_id} in {top_combination.category} generated ₹{top_combination.category_revenue:,.2f}.')
print(f'4. High-value preferences: {top_preferred_category} is the preferred category for {top_preferred_category_count:,} high-value customers; {top_preferred_product} is the preferred product for {top_preferred_product_count:,}.')


## Business findings, limitations, and recommendations

### Findings

- The customer and product summary joins preserved all **5,000** sales rows and **5,000** distinct order IDs. The chosen keys therefore did not multiply transaction records in this dataset.
- Data-quality checks found **0** sales rows without a customer-summary match and **0** without a product-summary match; all 5,000 rows matched both analytical layers.
- The largest customer-category combination is **C0127 — Electronics**, with **₹1,493,500.00** in category revenue.
- Of the **497** Day 18 high-value customers, **495** have Electronics as their highest-revenue category. **Laptop** is the leading preferred product for **241** of them.

### Limitations

- `customer_summary` and `product_summary` are derived from the same sales history, so their values are descriptive rather than independent measures.
- Preference is based on historical net-sales revenue, not customer surveys, profitability, inventory availability, or future demand.
- This validation checks matching keys and row preservation; it does not establish whether the source transaction values themselves are correct.

### Recommendations

- Keep the two-key (`product`, `category`) join whenever product-summary data is added to transaction analysis, and repeat row-count checks after future schema changes.
- Use the Electronics and Laptop preference pattern to guide high-value-customer merchandising, while validating margin and stock before acting.
- Add the unmatched-record checks to future data-load reviews so missing summary coverage is detected before customer reporting is published.

In [ ]:
# Save important Day 19 analytical outputs for reuse outside the notebook.
row_counts.to_csv('../data/processed/day19_join_row_counts.csv', index=False)
customer_category_performance.to_csv('../data/processed/day19_customer_category_performance.csv', index=False)
customer_product_performance.to_csv('../data/processed/day19_customer_product_performance.csv', index=False)
high_value_customer_preferences.to_csv('../data/processed/day19_high_value_customer_preferences.csv', index=False)
join_data_quality_validation.to_csv('../data/processed/day19_join_data_quality_validation.csv', index=False)

print('Saved 5 Day 19 processed outputs to ../data/processed/')
connection.close()
